## **Lap & Weather Integration**

##### **Imports**

In [160]:
import pandas as pd

from pathlib import Path

In [161]:
# Import and combine all f1 lap data
weather_directory = Path('../data/ff1_weather_data')
weather_files = weather_directory.glob('weather_data_*.csv')

weather_data = pd.concat(
    [pd.read_csv(file) for file in weather_files],
    ignore_index=True
)

weather_data.head()

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,Location,Year
0,0 days 00:00:57.060000,24.1,36.2,997.1,False,38.2,294,3.0,Melbourne,2018
1,0 days 00:01:57.078000,24.0,36.3,997.1,False,38.6,273,1.4,Melbourne,2018
2,0 days 00:02:57.090000,24.0,36.3,997.1,False,38.6,273,1.4,Melbourne,2018
3,0 days 00:03:57.106000,23.9,37.2,997.0,False,38.7,287,2.3,Melbourne,2018
4,0 days 00:04:57.121000,24.2,35.8,997.1,False,38.7,309,3.5,Melbourne,2018


#### **Weather Cleaning**

In [162]:
# Standardize 'Time' column type to timedelta
weather_data['Time'] = pd.to_timedelta(weather_data['Time'])

In [163]:
# Normalize location names 
weather_data['Location'] = weather_data['Location'].replace({
    'Monte Carlo': 'Monaco',
    'Marina Bay': 'Singapore',
    'Miami Gardens': 'Miami',
    'Yas Island': 'Yas Marina'
})

In [164]:
# Load event data
event_data = pd.read_csv('../data/event_location_data.csv')

# Set 'StartTime' to datetime and sort values
event_data['StartTime'] = pd.to_datetime(event_data['StartTime'])
event_data.sort_values(['Year', 'Location', 'StartTime'])

event_data['RaceSeq'] = event_data.groupby(['Year', 'Location']).cumcount()

In [165]:
# (Current row time - next row time < 0) = next race
new_race = (
    weather_data
    .groupby(['Year', 'Location'])['Time']
    .diff() < pd.Timedelta(0)
).astype(int)

weather_data['RaceSeq'] = (
    new_race
    .groupby([weather_data['Year'], weather_data['Location']])
    .cumsum()
)

# Add event name to handle repeat locations
weather_data = weather_data.merge(
    event_data[['Year', 'Location', 'RaceSeq', 'EventName', 'StartTime']],
    on=['Year', 'Location', 'RaceSeq'],
    how='left'
)

# Add a UTC lap start time for merging with weather data
weather_data['TimeUTC'] = weather_data['Time'] + weather_data['StartTime']

weather_data = weather_data.drop(['RaceSeq', 'StartTime'], axis=1)


In [166]:
# Resampling function
def resample_func(weather):
    weather = weather.copy()
    weather.set_index('TimeUTC', inplace=True) # required for resample
    
    numeric_cols = ["AirTemp", "Humidity", "Pressure", "TrackTemp",
                    "WindDirection", "WindSpeed"]
    
    interpolate_df = weather[numeric_cols].resample("1s").interpolate()
    ffill_df = weather[["Rainfall"]].resample("1s").ffill()
    
    return interpolate_df.join(ffill_df)
    
# Round TimeUTC down to the nearest second
weather_data["TimeUTC"] = weather_data["TimeUTC"].dt.floor("1s")

# Resample weather data
weather_resampled = (
    weather_data
    .groupby(['Year', 'Location', 'EventName'])
    .apply(resample_func, include_groups=False)
    .reset_index()
)

In [170]:
weather_resampled.head()

,Year,Location,EventName,TimeUTC,AirTemp,Humidity,Pressure,TrackTemp,WindDirection,WindSpeed,Rainfall
0,2018,Austin,United States Grand Prix,2018-10-21 18:10:39,20.700,41.300000,1007.000000,27.600000,93.000000,2.400000,False
1,2018,Austin,United States Grand Prix,2018-10-21 18:10:40,20.695,41.303333,1006.998333,27.596667,92.933333,2.436667,False
2,2018,Austin,United States Grand Prix,2018-10-21 18:10:41,20.690,41.306667,1006.996667,27.593333,92.866667,2.473333,False
3,2018,Austin,United States Grand Prix,2018-10-21 18:10:42,20.685,41.310000,1006.995000,27.590000,92.800000,2.510000,False
4,2018,Austin,United States Grand Prix,2018-10-21 18:10:43,20.680,41.313333,1006.993333,27.586667,92.733333,2.546667,False
